# Compute Secure Code Pass Rates for CodeSecEval

This notebook evaluates each task's **Secure Code** against three test fields:

- `Test`: the full canonical test suite
- `Test-SP`: the security-oriented test suite
- `Test-FP`: the functional-only test suite

It reports pass counts and pass rates for `SecEvalBase`, `SecEvalPlus`, and the combined dataset.

The notebook only reads the dataset files and executes the provided `Secure Code` and test fields. It does **not** modify the dataset content.


## 1. Configure input files

By default, this notebook assumes the Hugging Face dataset structure:

```text
CodeSecEval/
├── data/
│   ├── SecEvalBase/
│   │   └── test.jsonl
│   └── SecEvalPlus/
│       └── test.jsonl
└── README.md
```

If your files are still JSON arrays, change the paths below to the corresponding `.json` files, e.g., `test(2).json` and `test(3).json`.


In [1]:
from pathlib import Path
import json
import os
import sys
import subprocess
import tempfile
from datetime import datetime

import pandas as pd

# Change these paths if your local file names are different.
DATA_FILES = {
    "SecEvalBase": Path("data/SecEvalBase/test.jsonl"),
    "SecEvalPlus": Path("data/SecEvalPlus/test.jsonl"),
}

# Alternative example if you want to run directly on JSON array files:
# DATA_FILES = {
#     "SecEvalBase": Path("test(2).json"),
#     "SecEvalPlus": Path("test(3).json"),
# }

TEST_FIELDS = ["Test", "Test-SP", "Test-FP"]
TIMEOUT_SECONDS = 5

# If True, the runner provides lightweight stubs for common optional packages
# such as flask, canvasapi, openai, yaml, etc. This is useful because many tests
# patch external dependencies with fakes. Set to False if you want a stricter
# environment that requires all imports to be installed.
ENABLE_IMPORT_STUBS = True


## 2. Load JSON or JSONL files

The loader supports both:

- JSON array files: `[ {...}, {...} ]`
- JSONL files: one JSON object per line


In [2]:
def load_records(path: Path):
    """Load either a JSON array file or a JSONL file."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")

    if path.suffix.lower() == ".jsonl":
        records = []
        with path.open("r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as e:
                    raise ValueError(f"Invalid JSONL at {path}, line {line_no}: {e}") from e
        return records

    if path.suffix.lower() == ".json":
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if not isinstance(data, list):
            raise TypeError(f"Expected JSON array in {path}")
        return data

    raise ValueError(f"Unsupported file type: {path}")


datasets = {}
for subset, path in DATA_FILES.items():
    datasets[subset] = load_records(path)
    print(f"Loaded {subset}: {len(datasets[subset])} records from {path}")

print("Total:", sum(len(v) for v in datasets.values()))


Loaded SecEvalBase: 115 records from data/SecEvalBase/test.jsonl
Loaded SecEvalPlus: 140 records from data/SecEvalPlus/test.jsonl
Total: 255


## 3. Define isolated execution runner

Each task/test pair is executed in a separate Python subprocess with a timeout. This avoids one hanging or crashing task from stopping the whole evaluation.

The runner executes:

1. `Secure Code`
2. extracts the candidate function using `Entry_Point`
3. executes the selected test field: `Test`, `Test-SP`, or `Test-FP`
4. calls `check(candidate)`


In [3]:
RUNNER_CODE = r'''
import json
import sys
import traceback
import types
import importlib.abc
import importlib.util
import importlib.machinery


class Dummy:
    """
    Generic placeholder object used when an optional third-party
    dependency is unavailable.

    It supports common operations so that importing and defining
    candidate functions does not immediately fail.
    """

    def __init__(self, *args, **kwargs):
        pass

    def __call__(self, *args, **kwargs):
        return Dummy()

    def __getattr__(self, name):
        return Dummy()

    def __getitem__(self, key):
        return Dummy()

    def __setitem__(self, key, value):
        pass

    def __iter__(self):
        return iter([])

    def __len__(self):
        return 0

    def __bool__(self):
        return False

    def __enter__(self):
        return self

    def __exit__(self, *args):
        return False


class DummyModule(types.ModuleType):
    """
    Module placeholder that lazily creates Dummy attributes.
    """

    def __getattr__(self, name):
        value = Dummy()
        setattr(self, name, value)
        return value


# Only libraries that can reasonably use import-time fallback stubs
# should appear here.
#
# yaml is deliberately excluded because YAML-related tests require
# the real PyYAML implementation, especially yaml.safe_load().
ALLOWED_STUB_TOPLEVELS = {
    "flask",
    "canvasapi",
    "openai",
    "jwt",
    "requests",
    "django",
    "Crypto",
    "cryptography",
    "paramiko",
    "pymysql",
    "mysql",
    "psycopg2",
    "bs4",
    "PIL",
    "Pillow",
    "boto3",
    "botocore",
}

class StubImporter(
    importlib.abc.MetaPathFinder,
    importlib.abc.Loader,
):
    """
    Fallback importer for selected optional third-party dependencies.

    This importer must be appended to sys.meta_path, not inserted at
    position 0. Therefore, normal Python import mechanisms always have
    the first opportunity to load a real installed module.
    """

    def find_spec(self, fullname, path=None, target=None):
        root = fullname.split(".")[0]

        if root not in ALLOWED_STUB_TOPLEVELS:
            return None

        # Defensive check: do not create a stub if PathFinder can locate
        # an actual installed module.
        try:
            real_spec = importlib.machinery.PathFinder.find_spec(
                fullname,
                path,
            )
        except Exception:
            real_spec = None

        if real_spec is not None:
            return None

        return importlib.util.spec_from_loader(
            fullname,
            self,
            is_package=True,
        )

    def create_module(self, spec):
        module = DummyModule(spec.name)
        module.__path__ = []
        module.__package__ = spec.name
        return module

    def exec_module(self, module):
        module_name = module.__name__

        if module_name == "flask":
            self._install_flask_stub(module)

        elif module_name == "canvasapi":
            module.Canvas = Dummy

        elif module_name == "openai":
            self._install_openai_stub(module)

        # Other allowed modules use DummyModule.__getattr__ dynamically.

    @staticmethod
    def _install_flask_stub(module):
        class DummyFlask:
            def __init__(self, *args, **kwargs):
                self.config = {}

            def route(self, *args, **kwargs):
                def decorator(func):
                    return func
                return decorator

            def before_request(self, func):
                return func

            def after_request(self, func):
                return func

            def teardown_request(self, func):
                return func

            def errorhandler(self, *args, **kwargs):
                def decorator(func):
                    return func
                return decorator

            def context_processor(self, func):
                return func

            def template_filter(self, *args, **kwargs):
                def decorator(func):
                    return func
                return decorator

            def run(self, *args, **kwargs):
                return None

        class DummyResponse:
            def __init__(
                self,
                response=None,
                status=200,
                headers=None,
                mimetype=None,
                content_type=None,
            ):
                self.response = response
                self.data = response
                self.status_code = status
                self.headers = headers or {}
                self.mimetype = mimetype
                self.content_type = content_type

            def get_json(self, *args, **kwargs):
                return self.data

            def set_cookie(self, *args, **kwargs):
                return None

            def delete_cookie(self, *args, **kwargs):
                return None

        def make_response(*args):
            if not args:
                return DummyResponse()

            if len(args) == 1:
                value = args[0]

                if isinstance(value, DummyResponse):
                    return value

                return DummyResponse(value)

            response = args[0]
            status = args[1] if len(args) >= 2 else 200
            headers = args[2] if len(args) >= 3 else None

            return DummyResponse(
                response=response,
                status=status,
                headers=headers,
            )

        def jsonify(*args, **kwargs):
            if kwargs:
                data = kwargs
            elif len(args) == 1:
                data = args[0]
            else:
                data = list(args)

            response = DummyResponse(data)
            response.is_json = True
            return response

        def redirect(location, code=302, *args, **kwargs):
            response = DummyResponse(status=code)
            response.location = location
            return response

        def abort(code, *args, **kwargs):
            raise RuntimeError(f"Flask abort: {code}")

        def render_template(*args, **kwargs):
            return ""

        module.Flask = DummyFlask
        module.Response = DummyResponse
        module.make_response = make_response
        module.jsonify = jsonify
        module.redirect = redirect
        module.abort = abort
        module.render_template = render_template

        module.request = types.SimpleNamespace(
            args={},
            form={},
            json=None,
            files={},
            values={},
            headers={},
            cookies={},
            method="GET",
            path="/",
            url="http://localhost/",
            remote_addr="127.0.0.1",
            get_json=lambda *args, **kwargs: None,
        )

        module.session = {}
        module.g = types.SimpleNamespace()
        module.current_app = types.SimpleNamespace(config={})

    @staticmethod
    def _install_openai_stub(module):
        module.api_key = None

        class DummyEmbedding:
            @staticmethod
            def create(*args, **kwargs):
                return {
                    "data": [
                        {
                            "embedding": []
                        }
                    ]
                }

        class DummyCompletion:
            @staticmethod
            def create(*args, **kwargs):
                return {
                    "choices": [
                        {
                            "text": ""
                        }
                    ]
                }

        class DummyChatCompletion:
            @staticmethod
            def create(*args, **kwargs):
                return {
                    "choices": [
                        {
                            "message": {
                                "role": "assistant",
                                "content": "",
                            }
                        }
                    ]
                }

        module.Embedding = DummyEmbedding
        module.Completion = DummyCompletion
        module.ChatCompletion = DummyChatCompletion


def install_common_stubs(enable=True):
    """
    Install the fallback importer.

    Real installed modules always take priority because this importer is
    appended to sys.meta_path.
    """
    if not enable:
        return

    if not any(
        isinstance(finder, StubImporter)
        for finder in sys.meta_path
    ):
        sys.meta_path.append(StubImporter())


def main():
    record_path = sys.argv[1]
    test_field = sys.argv[2]
    enable_import_stubs = sys.argv[3].lower() == "true"

    result = {
        "passed": False,
        "error_type": None,
        "error_message": None,
        "traceback": None,
    }

    try:
        install_common_stubs(enable_import_stubs)

        with open(record_path, "r", encoding="utf-8") as f:
            record = json.load(f)

        required_fields = [
            "Secure Code",
            "Entry_Point",
            test_field,
        ]

        for key in required_fields:
            if key not in record:
                raise KeyError(f"Missing field: {key}")

        if not isinstance(record["Secure Code"], str):
            raise TypeError("Secure Code must be a string")

        if not isinstance(record["Entry_Point"], str):
            raise TypeError("Entry_Point must be a string")

        if not isinstance(record[test_field], str):
            raise TypeError(f"{test_field} must be a string")

        namespace = {
            "__name__": "codesec_eval_candidate",
            "__package__": None,
            "__builtins__": __builtins__,
        }

        # Execute Secure Code.
        exec(
            compile(
                record["Secure Code"],
                "<secure_code>",
                "exec",
            ),
            namespace,
        )

        # Extract candidate function using Entry_Point.
        entry_point = record["Entry_Point"]

        if entry_point not in namespace:
            raise KeyError(
                "Entry point not found after executing Secure Code: "
                f"{entry_point}"
            )

        candidate = namespace[entry_point]

        if not callable(candidate):
            raise TypeError(
                f"Entry point is not callable: {entry_point}"
            )

        # Execute the selected test field.
        #
        # The test uses the same namespace so that test code can patch
        # candidate.__globals__, such as replacing Canvas or request.
        exec(
            compile(
                record[test_field],
                f"<{test_field}>",
                "exec",
            ),
            namespace,
        )

        if "check" not in namespace:
            raise KeyError(
                f"No check(candidate) function found in field: {test_field}"
            )

        check_function = namespace["check"]

        if not callable(check_function):
            raise TypeError(
                f"check is not callable in field: {test_field}"
            )

        # Run the actual test.
        check_function(candidate)

        result["passed"] = True

    except BaseException as e:
        result["passed"] = False
        result["error_type"] = type(e).__name__
        result["error_message"] = str(e)
        result["traceback"] = traceback.format_exc()

    # Keep the structured result as the final stdout line.
    print(json.dumps(result, ensure_ascii=False))


if __name__ == "__main__":
    main()
'''

In [ ]:
def run_secure_code_test(record, test_field, timeout_seconds=5, enable_import_stubs=True):
    """Run one record's Secure Code against one test field in a subprocess."""
    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        record_path = tmpdir / "record.json"
        runner_path = tmpdir / "runner.py"

        record_path.write_text(json.dumps(record, ensure_ascii=False), encoding="utf-8")
        runner_path.write_text(RUNNER_CODE, encoding="utf-8")

        try:
            completed = subprocess.run(
                [
                    sys.executable,
                    str(runner_path),
                    str(record_path),
                    test_field,
                    str(enable_import_stubs),
                ],
                cwd=str(tmpdir),
                capture_output=True,
                text=True,
                timeout=timeout_seconds,
            )
        except subprocess.TimeoutExpired:
            return {
                "passed": False,
                "error_type": "TimeoutExpired",
                "error_message": f"Timed out after {timeout_seconds} seconds",
                "traceback": "",
                "returncode": None,
                "stdout": "",
                "stderr": "",
            }

        stdout = completed.stdout.strip()
        try:
            result = json.loads(stdout.splitlines()[-1])
        except Exception:
            result = {
                "passed": False,
                "error_type": "RunnerOutputError",
                "error_message": "Could not parse runner JSON output",
                "traceback": "",
            }

        result["returncode"] = completed.returncode
        result["stdout"] = completed.stdout
        result["stderr"] = completed.stderr
        return result


## 4. Run evaluation

This cell may take a few minutes because it runs `number_of_records × 3` subprocesses.


In [4]:
rows = []
started_at = datetime.now()

for subset, records in datasets.items():
    print(f"\nEvaluating {subset}: {len(records)} records")
    for idx, record in enumerate(records, start=1):
        task_id = record.get("ID", f"{subset}_{idx}")
        entry_point = record.get("Entry_Point", None)

        if idx == 1 or idx % 20 == 0 or idx == len(records):
            print(f"  {idx}/{len(records)}: {task_id}")

        for test_field in TEST_FIELDS:
            result = run_secure_code_test(
                record=record,
                test_field=test_field,
                timeout_seconds=TIMEOUT_SECONDS,
                enable_import_stubs=ENABLE_IMPORT_STUBS,
            )

            rows.append({
                "subset": subset,
                "id": task_id,
                "entry_point": entry_point,
                "test_field": test_field,
                "passed": bool(result.get("passed")),
                "error_type": result.get("error_type"),
                "error_message": result.get("error_message"),
                "traceback": result.get("traceback"),
                "returncode": result.get("returncode"),
                "stderr": result.get("stderr"),
            })

finished_at = datetime.now()
df = pd.DataFrame(rows)
print(f"\nFinished in {finished_at - started_at}")
df.head()



Evaluating SecEvalBase: 115 records
  1/115: CWE-502_codeql_1.py


NameError: name 'run_secure_code_test' is not defined

## 5. Summarize pass counts and pass rates


In [ ]:
summary_by_subset = (
    df.groupby(["subset", "test_field"], dropna=False)
      .agg(total=("passed", "size"), passed=("passed", "sum"))
      .reset_index()
)
summary_by_subset["failed"] = summary_by_subset["total"] - summary_by_subset["passed"]
summary_by_subset["pass_rate"] = summary_by_subset["passed"] / summary_by_subset["total"]
summary_by_subset["pass_rate_percent"] = (summary_by_subset["pass_rate"] * 100).round(2)

summary_overall = (
    df.groupby(["test_field"], dropna=False)
      .agg(total=("passed", "size"), passed=("passed", "sum"))
      .reset_index()
)
summary_overall.insert(0, "subset", "CodeSecEval")
summary_overall["failed"] = summary_overall["total"] - summary_overall["passed"]
summary_overall["pass_rate"] = summary_overall["passed"] / summary_overall["total"]
summary_overall["pass_rate_percent"] = (summary_overall["pass_rate"] * 100).round(2)

summary = pd.concat([summary_by_subset, summary_overall], ignore_index=True)
summary = summary[["subset", "test_field", "total", "passed", "failed", "pass_rate_percent"]]
summary
